In [23]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

print("Project folder:", PROJECT_DIR)
print("Data folder exists:", DATA_DIR.exists())
print("Files:", list(DATA_DIR.iterdir()))

Project folder: c:\Hackathon\Tyre-Retail-Analytics
Data folder exists: True
Files: [WindowsPath('c:/Hackathon/Tyre-Retail-Analytics/data/About Dataset.md'), WindowsPath('c:/Hackathon/Tyre-Retail-Analytics/data/current_stocks.csv'), WindowsPath('c:/Hackathon/Tyre-Retail-Analytics/data/sales.csv'), WindowsPath('c:/Hackathon/Tyre-Retail-Analytics/data/supplier.csv')]


In [24]:
import pandas as pd
from pandas import DataFrame

# Current Stock Cleaning

In [25]:
curr_stocks = pd.read_csv(DATA_DIR / "current_stocks.csv")
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost,Last Restock Date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,2766.0,2026-01-20
1,P0222,BRAND_C,145/80 R 12 BY-801,364,2876.0,2026-01-20
2,P0258,BRAND_C,155/65 R 13 BY-801,191,2976.0,2026-01-20
3,P0228,BRAND_C,145/80 R 13 BY-801,26,3067.0,2025-12-22
4,P0168,BRAND_A,145/80 R 13 TR203,167,3245.0,2025-06-16


In [26]:
curr_stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 253 entries, 0 to 252
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Product ID         253 non-null    str    
 1   Brand              253 non-null    str    
 2   Product Name       253 non-null    str    
 3   Current Stock Qty  253 non-null    int64  
 4   Unit Cost          253 non-null    float64
 5   Last Restock Date  253 non-null    str    
dtypes: float64(1), int64(1), str(4)
memory usage: 12.0 KB


In [27]:
curr_stocks['Product ID'].nunique()

253

In [28]:
curr_stocks = curr_stocks.rename(columns= {'Unit Cost (INR)':'Unit Cost'})
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost,Last Restock Date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,2766.0,2026-01-20
1,P0222,BRAND_C,145/80 R 12 BY-801,364,2876.0,2026-01-20
2,P0258,BRAND_C,155/65 R 13 BY-801,191,2976.0,2026-01-20
3,P0228,BRAND_C,145/80 R 13 BY-801,26,3067.0,2025-12-22
4,P0168,BRAND_A,145/80 R 13 TR203,167,3245.0,2025-06-16


1. Handling date format inconsistencies

In [29]:
#Check delimiters
date_delimiters = curr_stocks['Last Restock Date'].str.replace(r'[a-zA-Z0-9]', '', regex=True).value_counts()
print(date_delimiters)

Last Restock Date
--    253
Name: count, dtype: int64


In [30]:
#Checks varying length
date_len = curr_stocks['Last Restock Date'].str.len().value_counts()
date_len

Last Restock Date
10    253
Name: count, dtype: int64

In [31]:
#Group dates with varying size and formats
for length, group in curr_stocks.groupby(curr_stocks['Last Restock Date'].str.len()):
    print(f"\n--- Length: {length} (Total rows: {len(group)}) ---")

    # Check which delimiter this length uses
    has_slash = group['Last Restock Date'].str.contains('/').any()
    has_dash = group['Last Restock Date'].str.contains('-').any()

    print(f"Contains Slashes: {has_slash} | Contains Dashes: {has_dash}")
    print("Sample rows:")
    print(group['Last Restock Date'].head(3).to_string(index=False))


--- Length: 10 (Total rows: 253) ---
Contains Slashes: False | Contains Dashes: True
Sample rows:
2026-01-20
2026-01-20
2026-01-20


In [32]:
# Standardizing date format
# Step 1: Handle the slash formats (Lengths 9 and 10)
# %m handles both 1-digit and 2-digit months seamlessly
slash_parsed = pd.to_datetime(
    curr_stocks['Last Restock Date'], format="%m/%d/%Y", errors="coerce"
)

# Step 2: Handle the short dash format (Length 8)
# %y (lowercase) correctly reads 2-digit years like '25' as 2025
dash_parsed = pd.to_datetime(
    curr_stocks['Last Restock Date'], format="%m-%d-%y", errors="coerce"
)

# Step 3: Combine them together (if slash fails, use dash)
curr_stocks["final_clean_date"] = slash_parsed.fillna(dash_parsed)

print(curr_stocks)

    Product ID    Brand                   Product Name  Current Stock Qty  \
0        P0224  BRAND_C             145/70 R 12 BY-801                 15   
1        P0222  BRAND_C             145/80 R 12 BY-801                364   
2        P0258  BRAND_C             155/65 R 13 BY-801                191   
3        P0228  BRAND_C             145/80 R 13 BY-801                 26   
4        P0168  BRAND_A              145/80 R 13 TR203                167   
..         ...      ...                            ...                ...   
248      P0001  BRAND_D     1000 R 20/18 PR CT23 (SET)                719   
249      P0013  BRAND_D    315/80 R 22.5 /20 PR (CD01)                 84   
250      P0011  BRAND_D    315/80 R 22.5 /20 PR (CT23)                 24   
251      P0166  BRAND_A               145/80 R 12 TR17                 76   
252      P0119  BRAND_A  265/75 R 16/ 10PR LT TR29 OWL                 12   

     Unit Cost Last Restock Date final_clean_date  
0       2766.0        2

In [33]:
curr_stocks = curr_stocks.drop(columns=['Last Restock Date'])
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost,final_clean_date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,2766.0,NaT
1,P0222,BRAND_C,145/80 R 12 BY-801,364,2876.0,NaT
2,P0258,BRAND_C,155/65 R 13 BY-801,191,2976.0,NaT
3,P0228,BRAND_C,145/80 R 13 BY-801,26,3067.0,NaT
4,P0168,BRAND_A,145/80 R 13 TR203,167,3245.0,NaT


In [34]:
print(curr_stocks['final_clean_date'].max())
print(curr_stocks['final_clean_date'].min())

NaT
NaT


In [35]:
curr_stocks = curr_stocks.rename(columns={'final_clean_date':'Last Restock Date'})
curr_stocks.head()

,Product ID,Brand,Product Name,Current Stock Qty,Unit Cost,Last Restock Date
0,P0224,BRAND_C,145/70 R 12 BY-801,15,2766.0,NaT
1,P0222,BRAND_C,145/80 R 12 BY-801,364,2876.0,NaT
2,P0258,BRAND_C,155/65 R 13 BY-801,191,2976.0,NaT
3,P0228,BRAND_C,145/80 R 13 BY-801,26,3067.0,NaT
4,P0168,BRAND_A,145/80 R 13 TR203,167,3245.0,NaT


2. Changing column data type from object to numerical

In [36]:
# Cast to string to safely use string methods
unit_cost_str = curr_stocks['Unit Cost'].astype(str)

In [37]:
unit_cost_str = unit_cost_str.str.replace(',', '', regex=False)
unit_cost_str = unit_cost_str.str.strip()

In [38]:
# Changing data type to numerical
curr_stocks['Unit Cost'] = pd.to_numeric(unit_cost_str, errors='coerce')

In [39]:
curr_stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 253 entries, 0 to 252
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   Product ID         253 non-null    str          
 1   Brand              253 non-null    str          
 2   Product Name       253 non-null    str          
 3   Current Stock Qty  253 non-null    int64        
 4   Unit Cost          253 non-null    float64      
 5   Last Restock Date  0 non-null      datetime64[s]
dtypes: datetime64[s](1), float64(1), int64(1), str(3)
memory usage: 12.0 KB


In [40]:
curr_stocks.to_csv('current_stocks.csv', index=False)

# Sales Table Cleaning

In [41]:
sales = pd.read_csv(DATA_DIR / "sales.csv")
sales.head()

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price,Revenue,Transaction Type,Price Level
0,2024-04-08,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,15124.0,60496.0,Sale,A
1,2024-04-08,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,19658.0,19658.0,Sale,X
2,2024-04-08,BRAND_A,P0207,145 R 12C / 8PR TR06,2,7068.0,14136.0,Sale,A
3,2024-04-08,BRAND_B,P0378,195/55 R 16 HB-2,2,5158.0,10315.0,Sale,A
4,2024-04-08,BRAND_B,P0382,225/55 R 16 HB-2,4,16209.0,64836.0,Sale,A


In [42]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 16999 entries, 0 to 16998
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              16999 non-null  str    
 1   Brand             16999 non-null  str    
 2   Product ID        16999 non-null  str    
 3   Product Name      16999 non-null  str    
 4   Units Sold        16999 non-null  int64  
 5   Unit Price        16999 non-null  float64
 6   Revenue           16999 non-null  float64
 7   Transaction Type  16999 non-null  str    
 8   Price Level       16999 non-null  str    
dtypes: float64(2), int64(1), str(6)
memory usage: 1.2 MB


1. Handiling numerical columns

In [43]:
# Changing column names
sales = sales.rename(columns={'Revenue (INR)':'Revenue','Unit Price (INR)':'Unit Price'})

In [44]:
# Cast to string to safely use string methods
revenue_str = sales['Revenue'].astype(str)
unit_price_str = sales['Unit Price'].astype(str)

In [45]:
# 2. Remove commas, currency symbols, and extra spaces
revenue_str = revenue_str.str.replace(',', '', regex=False)
revenue_str = revenue_str.str.strip()

unit_price_str = unit_price_str.str.replace(',', '', regex=False)
unit_price_str = unit_price_str.str.strip()

In [46]:
# Changing to numeric column
sales['Revenue'] = pd.to_numeric(revenue_str, errors='coerce')
sales

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price,Revenue,Transaction Type,Price Level
0,2024-04-08,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,15124.0,60496.0,Sale,A
1,2024-04-08,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,19658.0,19658.0,Sale,X
2,2024-04-08,BRAND_A,P0207,145 R 12C / 8PR TR06,2,7068.0,14136.0,Sale,A
3,2024-04-08,BRAND_B,P0378,195/55 R 16 HB-2,2,5158.0,10315.0,Sale,A
4,2024-04-08,BRAND_B,P0382,225/55 R 16 HB-2,4,16209.0,64836.0,Sale,A
...,...,...,...,...,...,...,...,...,...
16994,2025-12-09,BRAND_C,P0220,295/80 R 22.5 /18 PR (VS788),-2,39400.0,-78800.0,Return,A
16995,2026-02-23,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,40200.0,-160800.0,Return,A
16996,2026-02-28,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,40200.0,-160800.0,Return,A
16997,2025-12-18,BRAND_C,P0326,825 R 16/16 PR LO301 (SET),-2,29160.0,-58320.0,Return,A


In [47]:
sales['Unit Price'] = pd.to_numeric(unit_price_str, errors='coerce')
sales

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price,Revenue,Transaction Type,Price Level
0,2024-04-08,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,15124.0,60496.0,Sale,A
1,2024-04-08,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,19658.0,19658.0,Sale,X
2,2024-04-08,BRAND_A,P0207,145 R 12C / 8PR TR06,2,7068.0,14136.0,Sale,A
3,2024-04-08,BRAND_B,P0378,195/55 R 16 HB-2,2,5158.0,10315.0,Sale,A
4,2024-04-08,BRAND_B,P0382,225/55 R 16 HB-2,4,16209.0,64836.0,Sale,A
...,...,...,...,...,...,...,...,...,...
16994,2025-12-09,BRAND_C,P0220,295/80 R 22.5 /18 PR (VS788),-2,39400.0,-78800.0,Return,A
16995,2026-02-23,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,40200.0,-160800.0,Return,A
16996,2026-02-28,BRAND_C,P0221,295/80 R 22.5 /18 PR (VS806),-4,40200.0,-160800.0,Return,A
16997,2025-12-18,BRAND_C,P0326,825 R 16/16 PR LO301 (SET),-2,29160.0,-58320.0,Return,A


2. Handiling Date column

In [48]:
sales.head()

,Date,Brand,Product ID,Product Name,Units Sold,Unit Price,Revenue,Transaction Type,Price Level
0,2024-04-08,BRAND_A,P0102,215/70 R 16 TR28 OWL,4,15124.0,60496.0,Sale,A
1,2024-04-08,BRAND_A,P0144,33 x 12.5 R 15/ 6PR OWL TR28,1,19658.0,19658.0,Sale,X
2,2024-04-08,BRAND_A,P0207,145 R 12C / 8PR TR06,2,7068.0,14136.0,Sale,A
3,2024-04-08,BRAND_B,P0378,195/55 R 16 HB-2,2,5158.0,10315.0,Sale,A
4,2024-04-08,BRAND_B,P0382,225/55 R 16 HB-2,4,16209.0,64836.0,Sale,A


In [49]:
#Check delimiters
date_delimiters = sales['Date'].str.replace(r'[a-zA-Z0-9]', '', regex=True).value_counts()
print(date_delimiters)

Date
--    16999
Name: count, dtype: int64


In [64]:
import pandas as pd
from pathlib import Path

# Load the original sales CSV again
project_dir = Path.cwd()
sales = pd.read_csv(project_dir / "data" / "sales.csv")

# Check original date values
print("Original dates:")
print(sales["Date"].head())

# Convert dates from YYYY-MM-DD format
sales["Date"] = pd.to_datetime(
    sales["Date"].astype(str).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
)

# Verify the results
print("\nValid dates:", sales["Date"].notna().sum())
print("Invalid dates:", sales["Date"].isna().sum())
print("Earliest date:", sales["Date"].min())
print("Latest date:", sales["Date"].max())

Original dates:
0    2024-04-08
1    2024-04-08
2    2024-04-08
3    2024-04-08
4    2024-04-08
Name: Date, dtype: str

Valid dates: 16999
Invalid dates: 0
Earliest date: 2024-04-08 00:00:00
Latest date: 2026-03-31 00:00:00


In [65]:
output_dir = Path.cwd() / "cleaned_data"
output_dir.mkdir(exist_ok=True)

sales.to_csv(output_dir / "sales_cleaned.csv", index=False)

print("Corrected sales CSV saved successfully.")

Corrected sales CSV saved successfully.


In [51]:
print(sales['Date'].min())
print(sales['Date'].max())

NaT
NaT


In [52]:
sales.to_csv('sales.csv', index=False)

# **Supplier Table Analysis**

In [53]:
supplier = pd.read_csv(DATA_DIR / "supplier.csv")
supplier.head()

,Brand,Supplier ID,Supplier Name,Avg Lead Time (Days),Lead Time Std Dev (Days),Min Lead Time (Days),Max Lead Time (Days)
0,BRAND_A,SUP0001,Supplier 1,40,6.7,30,70
1,BRAND_D,SUP0002,Supplier 2,60,9.2,45,100
2,BRAND_C,SUP0003,Supplier 3,50,10.0,30,90
3,BRAND_B,SUP0004,Supplier 4,60,11.7,50,120


In [54]:
supplier.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Brand                     4 non-null      str    
 1   Supplier ID               4 non-null      str    
 2   Supplier Name             4 non-null      str    
 3   Avg Lead Time (Days)      4 non-null      int64  
 4   Lead Time Std Dev (Days)  4 non-null      float64
 5   Min Lead Time (Days)      4 non-null      int64  
 6   Max Lead Time (Days)      4 non-null      int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 356.0 bytes


In [55]:
supplier.to_csv('supplier.csv', index=False)

In [56]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 16999 entries, 0 to 16998
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype        
---  ------            --------------  -----        
 0   Date              0 non-null      datetime64[s]
 1   Brand             16999 non-null  str          
 2   Product ID        16999 non-null  str          
 3   Product Name      16999 non-null  str          
 4   Units Sold        16999 non-null  int64        
 5   Unit Price        16999 non-null  float64      
 6   Revenue           16999 non-null  float64      
 7   Transaction Type  16999 non-null  str          
 8   Price Level       16999 non-null  str          
dtypes: datetime64[s](1), float64(2), int64(1), str(5)
memory usage: 1.2 MB


In [57]:
curr_stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 253 entries, 0 to 252
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   Product ID         253 non-null    str          
 1   Brand              253 non-null    str          
 2   Product Name       253 non-null    str          
 3   Current Stock Qty  253 non-null    int64        
 4   Unit Cost          253 non-null    float64      
 5   Last Restock Date  0 non-null      datetime64[s]
dtypes: datetime64[s](1), float64(1), int64(1), str(3)
memory usage: 12.0 KB


In [58]:
from pathlib import Path

project_dir = Path.cwd()
data_dir = project_dir / "data"

print("CSV files in project folder:")
for file in project_dir.glob("*.csv"):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

print("\nCSV files in data folder:")
for file in data_dir.glob("*.csv"):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

CSV files in project folder:
current_stocks.csv - 11.61 KB
sales.csv - 1035.58 KB
supplier.csv - 0.28 KB

CSV files in data folder:
current_stocks.csv - 14.07 KB
sales.csv - 1201.58 KB
supplier.csv - 0.28 KB


In [59]:
import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
data_dir = project_dir / "data"

files = ["sales.csv", "current_stocks.csv", "supplier.csv"]

for filename in files:
    root_file = project_dir / filename
    data_file = data_dir / filename

    if root_file.exists() and data_file.exists():
        root_df = pd.read_csv(root_file)
        data_df = pd.read_csv(data_file)

        print(f"\n{filename}")
        print("Root folder shape:", root_df.shape)
        print("Data folder shape:", data_df.shape)
        print("Same columns:", list(root_df.columns) == list(data_df.columns))


sales.csv
Root folder shape: (16999, 9)
Data folder shape: (16999, 9)
Same columns: True

current_stocks.csv
Root folder shape: (253, 6)
Data folder shape: (253, 6)
Same columns: True

supplier.csv
Root folder shape: (4, 7)
Data folder shape: (4, 7)
Same columns: True


In [60]:
from pathlib import Path

project_dir = Path.cwd()
output_dir = project_dir / "cleaned_data"
output_dir.mkdir(exist_ok=True)

sales.to_csv(output_dir / "sales_cleaned.csv", index=False)
curr_stocks.to_csv(
    output_dir / "current_stocks_cleaned.csv", index=False
)
supplier.to_csv(output_dir / "supplier_cleaned.csv", index=False)

print("Cleaned datasets saved successfully:")
for file in output_dir.glob("*.csv"):
    print(file.name, "-", file.stat().st_size, "bytes")

Cleaned datasets saved successfully:
current_stocks_cleaned.csv - 11886 bytes
sales_cleaned.csv - 1060433 bytes
supplier_cleaned.csv - 289 bytes


In [61]:
print("Sales date data type:", sales["Date"].dtype)
print("Valid dates:", sales["Date"].notna().sum())
print("Invalid or missing dates:", sales["Date"].isna().sum())
print("Earliest date:", sales["Date"].min())
print("Latest date:", sales["Date"].max())

Sales date data type: datetime64[s]
Valid dates: 0
Invalid or missing dates: 16999
Earliest date: NaT
Latest date: NaT


In [62]:
import pandas as pd
from pathlib import Path

project_dir = Path.cwd()

original_sales = pd.read_csv(project_dir / "data" / "sales.csv")

print("Column names:")
print(original_sales.columns.tolist())

print("\nFirst 10 date values:")
print(original_sales["Date"].head(10).to_list())

print("\nOriginal data types:")
print(original_sales.dtypes)

Column names:
['Date', 'Brand', 'Product ID', 'Product Name', 'Units Sold', 'Unit Price', 'Revenue', 'Transaction Type', 'Price Level']

First 10 date values:
['2024-04-08', '2024-04-08', '2024-04-08', '2024-04-08', '2024-04-08', '2024-04-08', '2024-04-08', '2024-04-08', '2024-04-08', '2024-04-09']

Original data types:
Date                    str
Brand                   str
Product ID              str
Product Name            str
Units Sold            int64
Unit Price          float64
Revenue             float64
Transaction Type        str
Price Level             str
dtype: object
